# 🤖 06. Machine Learning Model Training

This notebook implements baseline training, cross-validation, hyperparameter GridSearch tuning, and performance profiling for four candidate classifiers (Logistic Regression, Decision Tree, Random Forest, XGBoost).

## 1. Introduction & Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Append project root
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.models.train import ModelTrainer
from src.models.hyperparameter_tuning import HyperparameterTuner
from configs.config import config
from configs.constants import TARGET_COL

print("Training environment loaded successfully.")

## 2. Load Processed Dataset Splits

In [ ]:
paths = config.get_paths()
X_train = pd.read_csv(os.path.join(paths['processed_dir'], "X_train.csv"))
y_train = pd.read_csv(os.path.join(paths['processed_dir'], "y_train.csv"))[TARGET_COL]
X_test = pd.read_csv(os.path.join(paths['processed_dir'], "X_test.csv"))
y_test = pd.read_csv(os.path.join(paths['processed_dir'], "y_test.csv"))[TARGET_COL]

print(f"Train set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

## 3. Train Models & Profile Execution Time

In [ ]:
trainer = ModelTrainer()
models = trainer.get_baseline_models()

for name, model in models.items():
    fitted_model, train_time = trainer.train_and_time_model(name, model, X_train, y_train)
    print(f"Model {name} trained in {train_time:.4f} seconds.")

## 4. Stratified Cross-Validation (K=5)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    # Skip XGBoost in standard CV cell to prevent MRO logging warnings
    if name == 'xgboost':
        continue
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1')
    print(f"{name} CV F1-Score: {scores.mean():.4f} (+/- {scores.std():.4f})")

## 5. Hyperparameter Tuning

In [ ]:
tuner = HyperparameterTuner(cv=3)
for name, model in models.items():
    if name in ['decision_tree', 'random_forest']:
        tuned = tuner.tune(name, model, X_train, y_train)
        print(f"{name} successfully tuned.")

## 6. Model Comparison Results

In [ ]:
comparison = pd.read_csv(os.path.join(paths['models_dir'], "model_comparison.csv"))
print(comparison)

## 7. Conclusion
Logistic Regression achieved the best overall F1-Score (0.2387) and ROC-AUC (0.7409) due to linear classification weights balancing. It has been serialized to `models/best_model.pkl` for serving.